# RAG: Baseline Retrieval vs. Cross-Encoder Re-Ranking

**Document:** *FIDE Laws of Chess* (Chapter 2 of the FIDE Arbiters' Manual 2026, 54 pages: Laws of Chess,
Appendices for Rapid/Blitz, Guidelines and the arbiters' commentary).
Source: https://arbiters.fide.com/wp-content/uploads/Publications/Manual/Arbiter-Manual-2026.pdf (PDF pages 14–67).

| Component | Choice |
|---|---|
| PDF parsing | `pypdf` |
| Bi-Encoder (dense retriever) | `sentence-transformers/all-MiniLM-L6-v2` (384-d) |
| Vector store | FAISS `IndexFlatIP` over L2-normalised vectors (= cosine similarity) |
| Cross-Encoder (re-ranker) | `cross-encoder/ms-marco-MiniLM-L-6-v2` |
| LLM | Gemini (`gemini-3.8-flash`, override with env var `GEMINI_MODEL`) via the `google-genai` SDK |

**Pipeline 1 (baseline):** query → bi-encoder → top-10 by cosine → top-3 → LLM → *Answer 1*
**Pipeline 2 (re-rank):** same top-10 → cross-encoder scores all 10 (query, chunk) pairs → re-sort → top-3 → same prompt → *Answer 2*

### Setup
```bash
pip install pypdf sentence-transformers faiss-cpu google-genai pandas
```
Put the Gemini key in a `.env` file next to the notebook (`GEMINI_API_KEY=...`, git-ignored) or set it as an
environment variable. Without a key, every retrieval / re-ranking /
latency cell still runs; only the LLM answer cells are skipped.

In [1]:
# ---- Imports and global configuration ----
import os, re, time, json, statistics
from pathlib import Path

import numpy as np
import pandas as pd
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder

PDF_PATH   = Path("data/FIDE_Laws_of_Chess_2026.pdf")
RESULTS    = Path("results"); RESULTS.mkdir(exist_ok=True)

BI_ENCODER    = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_MODEL     = os.environ.get("GEMINI_MODEL", "gemini-3.8-flash")   # gemini-2.5-flash is closed to new API keys

CHUNK_SIZE    = 700   # max characters per chunk
CHUNK_OVERLAP = 150   # characters shared by consecutive chunks (so a rule cut at a boundary survives in one of them)
TOP_K_RETRIEVE = 10   # candidates from the bi-encoder
TOP_K_CONTEXT  = 3    # chunks handed to the LLM

pd.set_option("display.max_colwidth", 120)

## 1. Load and clean the PDF
Each page is extracted with `pypdf`. Stand-alone page-number lines are removed, curly quotes normalised and
whitespace collapsed. The page number is kept with the text so every chunk can be traced back to its page.

In [2]:
def load_pages(path):
    # Returns a list of (page_number, cleaned_text) tuples.
    pages = []
    for i, page in enumerate(PdfReader(path).pages):
        text = page.extract_text() or ""
        text = (text.replace("\u2019", "'").replace("\u2018", "'")
                    .replace("\u201c", '"').replace("\u201d", '"'))
        lines = [l.strip() for l in text.splitlines()]
        # drop empty lines and lines that are only page numbers (e.g. "12" or "12 12")
        lines = [l for l in lines if l and not re.fullmatch(r"\d{1,3}( \d{1,3})?", l)]
        pages.append((i + 1, re.sub(r"[ \t]+", " ", "\n".join(lines))))
    return pages

pages = load_pages(PDF_PATH)
print(f"{len(pages)} pages, {sum(len(t) for _, t in pages):,} characters")
print(pages[19][1][:600])

54 pages, 112,827 characters
6.8 A flag is considered to have fallen when the arbiter observes the fact or when
either player has made a valid claim to that effect.
6.9 Except where one of Articles 5.1.1, 5.1.2, 5.2.1, 5.2.2, 5.2.3 applies, if a player
does not complete the prescribed number of moves in the allotted time, the game
is lost by that player. However, the game is drawn if the position is such that the
opponent cannot checkmate the player's king by any possible series o f legal
moves.
6.10 Chessclock setting:
6.10.1 Every indication given by the chessclock is considered to be conclusive
in the absence of any ev


## 2. Chunking
Fixed-size character windows (≈700 chars ≈ 110–130 words) with 150 characters of overlap. The window end is moved
back to the last sentence boundary (`". "`) when one exists in the second half of the window, so chunks rarely
stop mid-sentence. Chunking is done per page, so every chunk has exactly one page number.
**Both pipelines use this exact same list of chunks.**

In [3]:
def chunk_pages(pages, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    for page_no, text in pages:
        text = re.sub(r"\s*\n\s*", " ", text)
        start = 0
        while start < len(text):
            end = min(start + size, len(text))
            if end < len(text):
                cut = text.rfind(". ", start + size // 2, end)   # prefer ending on a sentence
                if cut != -1:
                    end = cut + 1
            chunks.append({"id": len(chunks), "page": page_no, "text": text[start:end].strip()})
            if end >= len(text):
                break
            start = max(end - overlap, start + 1)
            space = text.find(" ", start)                         # do not start in the middle of a word
            start = space + 1 if space != -1 and space < end else start
    return chunks

chunks = chunk_pages(pages)
chunk_texts = [c["text"] for c in chunks]
lengths = [len(t) for t in chunk_texts]
print(f"{len(chunks)} chunks | chars per chunk: mean {np.mean(lengths):.0f}, min {min(lengths)}, max {max(lengths)}")

238 chunks | chars per chunk: mean 587, min 173, max 700


## 3. Pipeline 1 — Bi-Encoder index (FAISS)
The bi-encoder embeds every chunk **independently** of any query. Vectors are L2-normalised, so the inner product
computed by `IndexFlatIP` is exactly cosine similarity. `IndexFlatIP` is an exact (brute-force) search, which is
fine for a few hundred chunks.

In [4]:
# Load the bi-encoder (a MiniLM transformer + mean pooling -> one 384-d vector per text).
bi_encoder = SentenceTransformer(BI_ENCODER)

t0 = time.perf_counter()
# Embed every chunk ONCE, offline. normalize_embeddings=True makes each vector unit length,
# so inner product == cosine similarity.
chunk_emb = bi_encoder.encode(chunk_texts, batch_size=64, normalize_embeddings=True,
                              convert_to_numpy=True, show_progress_bar=False).astype("float32")
# Exact (brute-force) inner-product index; fine for a few hundred chunks.
index = faiss.IndexFlatIP(chunk_emb.shape[1])
index.add(chunk_emb)
index_build_s = time.perf_counter() - t0
print(f"Indexed {index.ntotal} chunks, dim={chunk_emb.shape[1]}, offline indexing time {index_build_s:.2f}s")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Indexed 238 chunks, dim=384, offline indexing time 5.08s


In [5]:
def baseline_retrieve(query, k=TOP_K_RETRIEVE):
    # Encode the query with the same bi-encoder and return the top-k chunk ids with cosine scores.
    q = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, ids = index.search(q, k)          # cosine scores + chunk ids, best first
    return [{"chunk_id": int(i), "cos": float(s)} for i, s in zip(ids[0], scores[0])]

## 4. Pipeline 2 — Cross-Encoder re-ranking
The cross-encoder reads the query and a chunk **together** (`[CLS] query [SEP] chunk [SEP]`) and outputs one
relevance logit. It is applied only to the 10 baseline candidates; candidates are re-sorted by that score.

In [6]:
# Load the cross-encoder: it takes a (query, passage) PAIR and outputs a single relevance logit.
cross_encoder = CrossEncoder(CROSS_ENCODER)

def rerank(query, candidates):
    # Score all (query, chunk) pairs jointly and sort descending by cross-encoder score.
    pairs = [(query, chunks[c["chunk_id"]]["text"]) for c in candidates]
    ce_scores = cross_encoder.predict(pairs, show_progress_bar=False)   # one forward pass per pair
    out = []
    for base_rank, (c, s) in enumerate(zip(candidates, ce_scores), start=1):
        out.append({**c, "ce": float(s), "base_rank": base_rank})
    out.sort(key=lambda d: d["ce"], reverse=True)   # re-sort by cross-encoder score (higher = more relevant)
    for new_rank, d in enumerate(out, start=1):
        d["rerank_rank"] = new_rank
    return out

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

## 5. LLM generation (identical prompt for both pipelines)
The prompt instructs the model to answer only from the three supplied chunks. Temperature is 0 so that differences
between Answer 1 and Answer 2 come from the context, not from sampling.

In [7]:
PROMPT_TEMPLATE = '''You are an assistant answering questions about the FIDE Laws of Chess.
Answer the question using ONLY the context passages below. Quote article numbers where possible.
If the context does not contain the information needed, say "The provided context does not answer this."

Context:
{context}

Question: {question}
Answer (2-4 sentences):'''

# Same template for both pipelines; only the 3 context chunks change.
def build_prompt(question, chunk_ids):
    context = "\n\n".join(f"[Passage {n} | page {chunks[i]['page']}]\n{chunks[i]['text']}"
                          for n, i in enumerate(chunk_ids, start=1))
    return PROMPT_TEMPLATE.format(context=context, question=question)

# Read KEY=value lines from a local .env (git-ignored) so the key never has to be typed into the notebook.
if Path(".env").exists():
    for line in Path(".env").read_text().splitlines():
        if "=" in line and not line.lstrip().startswith("#"):
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"'))

GEMINI_KEY = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
llm_client = None
if GEMINI_KEY:
    from google import genai
    from google.genai import types
    llm_client = genai.Client(api_key=GEMINI_KEY)
    print("Gemini client ready:", LLM_MODEL)
else:
    print("No GEMINI_API_KEY found -> LLM answers will be skipped; retrieval/re-ranking still run.")

def generate(prompt, retries=4):
    if llm_client is None:
        return "[skipped: no GEMINI_API_KEY]"
    for attempt in range(retries):
        try:
            resp = llm_client.models.generate_content(
                model=LLM_MODEL, contents=prompt,
                config=types.GenerateContentConfig(temperature=0.0))
            return resp.text.strip()
        except Exception as e:                      # rate limits on the free tier -> back off and retry
            if getattr(e, "code", None) in (400, 401, 403, 404) or attempt == retries - 1:   # retrying won't fix these
                return f"[LLM error: {e}]"
            time.sleep(15 * (attempt + 1))

Gemini client ready: gemini-3.8-flash


## 6. The two end-to-end pipelines (with timing)
`time.perf_counter()` is recorded around each stage so the latency table later uses the same code path.

In [8]:
def pipeline_1(query, call_llm=True):
    t0 = time.perf_counter()
    cands = baseline_retrieve(query)
    t1 = time.perf_counter()
    top3 = [c["chunk_id"] for c in cands[:TOP_K_CONTEXT]]   # baseline: first 3 by cosine
    answer = generate(build_prompt(query, top3)) if call_llm else None
    t2 = time.perf_counter()
    return {"candidates": cands, "context_ids": top3, "answer": answer,
            "t_retrieve": t1 - t0, "t_llm": t2 - t1}

def pipeline_2(query, call_llm=True):
    t0 = time.perf_counter()
    cands = baseline_retrieve(query)          # same top-10 as pipeline 1
    t1 = time.perf_counter()
    reranked = rerank(query, cands)
    t2 = time.perf_counter()
    top3 = [c["chunk_id"] for c in reranked[:TOP_K_CONTEXT]]   # re-ranked: first 3 by cross-encoder score
    answer = generate(build_prompt(query, top3)) if call_llm else None
    t3 = time.perf_counter()
    return {"candidates": reranked, "context_ids": top3, "answer": answer,
            "t_retrieve": t1 - t0, "t_rerank": t2 - t1, "t_llm": t3 - t2}

## 7. Failure-mode stress tests
Each query has a **gold passage**: the chunk that actually contains the rule needed for a correct answer, identified by
a distinctive phrase from the rulebook (so the check does not depend on chunk ids). Because chunks overlap, a phrase
can occur in two chunks; the best-ranked one is reported.

Q1–Q7 are queries where the baseline puts the gold chunk at rank 4–10 (so it is **not** given to the LLM) and the
cross-encoder moves it into the top 3. L1–L2 are kept on purpose as **cases where re-ranking makes things worse**.

In [9]:
# Each entry: id, failure category, the query, a distinctive phrase from the gold chunk, and why the baseline fails.
FAILURE_QUERIES = [
    {"qid": "Q1", "category": "Keyword trap / distractor",
     "query": "How much extra time does the opponent get when a player makes an illegal move?",
     "gold": "7.5.5 After the action taken",
     "note": "Rapid appendix A.3 ('penalties ... one minute instead of two minutes') shares the 'penalty/minutes' vocabulary and ranks above Article 7.5.5."},
    {"qid": "Q2", "category": "Exception (rule stated elsewhere)",
     "query": "What is the penalty for an illegal move in rapid chess?",
     "gold": "A.3 The penalties mentioned",
     "note": "The general illegal-move rule (Art. 7.5) is retrieved; the rapid-chess override in Appendix A.3 is a different part of the document."},
    {"qid": "Q3", "category": "Negation / role limit",
     "query": "Which things is the arbiter not allowed to tell a player during the game?",
     "gold": "shall not indicate the number of moves",
     "note": "General chunks about the arbiter's powers rank above Art. 12.6, which lists what the arbiter must not do."},
    {"qid": "Q4", "category": "Exception / corner case",
     "query": "Is a player with less than five minutes required to keep recording the moves?",
     "gold": "not obliged to meet the requirements of Article 8.1.1",
     "note": "Top 3 are the rapid-chess appendix (A.2 says rapid players need not record) and a clock-delay chunk; the Art. 8.4 time-trouble exception for standard play is rank 4."},
    {"qid": "Q5", "category": "Corner case (buried at rank 5)",
     "query": "If I use two hands to castle illegally, is that counted as two illegal moves?",
     "gold": "they count as one (1) illegal move",
     "note": "Generic castling / two-hands chunks outrank the commentary that answers the question directly."},
    {"qid": "Q6", "category": "Negation / constraint",
     "query": "Can an illegal move in rapid chess be corrected after the opponent has replied?",
     "gold": "an illegal move cannot",
     "note": "Chunks about how illegal moves ARE corrected rank above A.5.2 which says when they CANNOT be."},
    {"qid": "Q7", "category": "Negation / prohibition",
     "query": "Can I keep my phone in my bag during the game?",
     "gold": "provided the device is completely switched off",
     "note": "Commentary about bag access and less severe penalties ranks above Art. 11.3.2.1, the condition (completely switched off) under which a phone in the bag is allowed."},
    {"qid": "L1", "category": "Limitation: article text demoted",
     "query": "Can a player write down his move on the scoresheet before playing it?",
     "gold": "It is forbidden to record the moves in advance",
     "note": "Art. 8.1.2 drops from rank 1 to 6. The commentary chunk repeating the rule in the question's own words ('before playing it') is promoted to rank 1, so the answer should survive, but the article itself is lost."},
    {"qid": "L2", "category": "Limitation: re-ranker demotes gold",
     "query": "Can a player castle with a rook that has already moved?",
     "gold": "With a rook that has already moved",
     "note": "Art. 3.8.2.1.2 drops out of the top 3 after re-ranking; touch-move chunks mentioning 'castle' and 'rook' are preferred."},
]

# Best rank (baseline or re-ranked, chosen by rank_key) of any candidate chunk that contains the gold phrase.
def gold_rank(cands, phrase, rank_key):
    ranks = [c[rank_key] for c in cands if phrase in chunks[c["chunk_id"]]["text"]]
    return min(ranks) if ranks else None

In [10]:
# Run retrieval + re-ranking for every failure query (no LLM yet) and build the rank-shift table.
rows, detail = [], {}
for fq in FAILURE_QUERIES:
    reranked = rerank(fq["query"], baseline_retrieve(fq["query"]))
    detail[fq["qid"]] = reranked
    b = gold_rank(reranked, fq["gold"], "base_rank")
    r = gold_rank(reranked, fq["gold"], "rerank_rank")
    gold_ids = [c["chunk_id"] for c in reranked if fq["gold"] in chunks[c["chunk_id"]]["text"]]
    rows.append({"qid": fq["qid"], "category": fq["category"], "gold chunk id(s)": gold_ids,
                 "baseline rank": b, "re-ranked rank": r,
                 "in baseline top-3": b is not None and b <= 3,
                 "in re-ranked top-3": r is not None and r <= 3})
rank_shift = pd.DataFrame(rows)
rank_shift.to_csv(RESULTS / "rank_shift_summary.csv", index=False)
rank_shift

,qid,category,gold chunk id(s),baseline rank,re-ranked rank,in baseline top-3,in re-ranked top-3
0,Q1,Keyword trap / distractor,[97],5,1,False,True
1,Q2,Exception (rule stated elsewhere),"[156, 157]",5,3,False,True
2,Q3,Negation / role limit,"[152, 151]",4,2,False,True
3,Q4,Exception / corner case,[105],4,1,False,True
4,Q5,Corner case (buried at rank 5),[99],5,1,False,True
5,Q6,Negation / constraint,[162],4,1,False,True
6,Q7,Negation / prohibition,[135],4,3,False,True
7,L1,Limitation: article text demoted,[103],1,6,True,False
8,L2,Limitation: re-ranker demotes gold,[32],3,5,True,False


In [11]:
# Full top-10 for each query: where every candidate started and where it ended up after re-ranking.
def show_candidates(qid):
    fq = next(f for f in FAILURE_QUERIES if f["qid"] == qid)
    df = pd.DataFrame([{
        "base_rank": c["base_rank"], "rerank_rank": c["rerank_rank"],
        "shift": c["base_rank"] - c["rerank_rank"], "chunk_id": c["chunk_id"],
        "page": chunks[c["chunk_id"]]["page"], "cosine": round(c["cos"], 3), "ce_score": round(c["ce"], 2),
        "gold": "★" if fq["gold"] in chunks[c["chunk_id"]]["text"] else "",
        "text": chunks[c["chunk_id"]]["text"][:110] + "…"} for c in detail[qid]]).sort_values("base_rank")
    print(f"{qid} [{fq['category']}] {fq['query']}\n  why: {fq['note']}")
    return df.reset_index(drop=True)

all_tables = []
for fq in FAILURE_QUERIES:
    t = show_candidates(fq["qid"]); t.insert(0, "qid", fq["qid"]); all_tables.append(t)
    display(t)
pd.concat(all_tables).to_csv(RESULTS / "rank_shift_full_top10.csv", index=False)

Q1 [Keyword trap / distractor] How much extra time does the opponent get when a player makes an illegal move?
  why: Rapid appendix A.3 ('penalties ... one minute instead of two minutes') shares the 'penalty/minutes' vocabulary and ranks above Article 7.5.5.


,qid,base_rank,rerank_rank,shift,chunk_id,page,cosine,ce_score,gold,text
0,Q1,1,9,-8,157,36,0.674,-2.37,,to write the moves. A.3 The penalties mentioned in Articles 7 and 9 of the Competitive Rules of Play shall be …
1,Q1,2,4,-2,164,37,0.661,1.85,,"an illegal position is still on the board, he/she shall declare the game drawn. A.5.5 The arbiter shall also c…"
2,Q1,3,5,-2,100,24,0.650,1.27,,"player who previously committed an illegal move, makes another one immediately after the game restarted (which…"
3,Q1,4,3,1,61,17,0.639,2.81,,amount of time added with each move. All these must be specified in advance. Sometimes the following happens: …
4,Q1,5,1,4,97,24,0.629,6.79,★,"7.5.5 After the action taken under Article 7.5.1, 7.5.2, 7.5.3 or 7.5.4 for the first completed illegal move b…"
5,Q1,6,6,0,91,23,0.615,-0.19,,7.5 Illegal moves: 7.5.1 An illegal move is completed once the player has pressed his/her clock. If during a g…
6,Q1,7,2,5,129,29,0.597,3.18,,shall take precedence. See comments to article 9.2.3.2 The right to claim a draw is returned on the next move …
7,Q1,8,8,0,92,23,0.583,-1.23,,"If the player has moved a pawn to the furthest distant rank, pressed the clock, but not replaced the pawn with…"
8,Q1,9,10,-1,34,11,0.573,-2.85,,3.10 Legal and illegal moves; illegal positions: 3.10.1 A move is legal when all the relevant requirements of …
9,Q1,10,7,3,99,24,0.572,-0.61,,The game shall then continue from this reinstated position. The player is forfeited if he/she completes two (2…


Q2 [Exception (rule stated elsewhere)] What is the penalty for an illegal move in rapid chess?
  why: The general illegal-move rule (Art. 7.5) is retrieved; the rapid-chess override in Appendix A.3 is a different part of the document.


,qid,base_rank,rerank_rank,shift,chunk_id,page,cosine,ce_score,gold,text
0,Q2,1,1,0,100,24,0.711,3.69,,"player who previously committed an illegal move, makes another one immediately after the game restarted (which…"
1,Q2,2,2,0,164,37,0.703,3.43,,"an illegal position is still on the board, he/she shall declare the game drawn. A.5.5 The arbiter shall also c…"
2,Q2,3,8,-5,34,11,0.692,0.96,,3.10 Legal and illegal moves; illegal positions: 3.10.1 A move is legal when all the relevant requirements of …
3,Q2,4,5,-1,35,11,0.688,2.86,,"and all 8 pawns are still on the board In Standard chess , if an arbiter observes an illegal position, he/she …"
4,Q2,5,3,2,156,36,0.658,2.94,★,APPENDICES Appendix A. Rapid chess A.1 A Rapid chess' game is one where either all the moves must be completed…
5,Q2,6,4,2,92,23,0.646,2.86,,"If the player has moved a pawn to the furthest distant rank, pressed the clock, but not replaced the pawn with…"
6,Q2,7,7,0,91,23,0.644,1.58,,7.5 Illegal moves: 7.5.1 An illegal move is completed once the player has pressed his/her clock. If during a g…
7,Q2,8,6,2,97,24,0.611,1.65,,"7.5.5 After the action taken under Article 7.5.1, 7.5.2, 7.5.3 or 7.5.4 for the first completed illegal move b…"
8,Q2,9,9,0,94,23,0.604,0.46,,"move' rule applies so the piece to be played should be, if possible, the one first touched, either the piece i…"
9,Q2,10,10,0,157,36,0.597,-2.23,★,to write the moves. A.3 The penalties mentioned in Articles 7 and 9 of the Competitive Rules of Play shall be …


Q3 [Negation / role limit] Which things is the arbiter not allowed to tell a player during the game?
  why: General chunks about the arbiter's powers rank above Art. 12.6, which lists what the arbiter must not do.


...

,qid,base_rank,rerank_rank,shift,chunk_id,page,cosine,ce_score,gold,text
0,Q7,1,1,0,143,32,0.525,3.24,,"not have access to the bag during the game, without the arbiter's permission (and he/she cannot take the bag w…"
1,Q7,2,2,0,142,32,0.502,0.87,,or an organizer to specify in advance (in the regulations of the event) a less severe penalty for a violation …
2,Q7,3,4,-1,139,32,0.428,-3.78,,who have finished their games shall be considered to be spectators. 11.5 It is forbidden to distract or annoy …
3,Q7,4,3,1,135,31,0.417,0.61,★,"chessboard. 11.3.2 During a game, a player is forbidden to have any electronic device not specifically approve…"
4,Q7,5,5,0,140,32,0.354,-4.85,,"phone (even switched off) is found with a player, his/her game is immediately lost and the opponent shall win.…"
5,Q7,6,7,-1,138,32,0.315,-7.54,,"11.3.3 The arbiter may require the player to allow his/her clothes, bags, other items or body to be inspected,…"
6,Q7,7,6,1,155,35,0.290,-5.33,,"person, 12.9.6 declaring the game to be lost by the offending player (the arbiter shall also decide the oppone…"
7,Q7,8,9,-1,141,32,0.279,-9.34,,Player B is absent. Immediately after player A made his/her first move his/her mobile rings. The arbiter decla…
8,Q7,9,8,1,134,31,0.274,-7.61,,having the move be allowed to leave the playing area. 11.2.3.3 a person who is neither a player nor arbiter be…
9,Q7,10,10,0,197,45,0.260,-9.39,,"his/her reply in the normal manner; if so, the envelope shall be handed to the arbiter for safekeeping and ope…"


L1 [Limitation: article text demoted] Can a player write down his move on the scoresheet before playing it?
  why: Art. 8.1.2 drops from rank 1 to 6. The commentary chunk repeating the rule in the question's own words ('before playing it') is promoted to rank 1, so the answer should survive, but the article itself is lost.


,qid,base_rank,rerank_rank,shift,chunk_id,page,cosine,ce_score,gold,text
0,L1,1,6,-5,103,25,0.725,5.68,★,'scoresheet' prescribed for the competition. 8.1.1.2 by entering moves on the FIDE certified 'electronic score…
1,L1,2,4,-2,107,25,0.672,6.12,,moves have been recorded. It is acceptable for a player to have a pen on his/her scoresheet but it should not …
2,L1,3,7,-4,105,25,0.670,5.62,,8.2 The scoresheet shall be visible to the arbiter through out the game. 8.3 The scoresheets are the property …
3,L1,4,1,3,106,25,0.640,7.35,,forbidden to record the move before playing it. Only in the case of a draw claim (Article 9.2. and 9.3) and ad…
4,L1,5,9,-4,102,25,0.601,3.12,,Article 8: The Recording of the Moves 8.1 How the moves shall be recorded: 8.1.1 In the course of play each pl…
5,L1,6,5,1,53,15,0.599,5.78,,"This rule is applicable, only if Article 9.1.1 (not to agree to a draw before a specified number of moves by e…"
6,L1,7,10,-3,104,25,0.586,2.98,,"shall be used only for recording the moves, the times of the clocks, offers of a draw, matters relating to a c…"
7,L1,8,3,5,109,26,0.565,6.28,,"on the chessboard. Provided it is that player's move, he/she may use h i s / h e r opponent's scoresheet, but …"
8,L1,9,2,7,112,26,0.561,6.48,,and request both players to update their scoresheets. When they are completed the arbiter will restart the clo…
9,L1,10,8,2,108,26,0.554,5.36,,"8.5 Incomplete scoresheets: 8.5.1 If neither player keeps score under Article 8.4, the arbiter or an assistant…"


L2 [Limitation: re-ranker demotes gold] Can a player castle with a rook that has already moved?
  why: Art. 3.8.2.1.2 drops out of the top 3 after re-ranking; touch-move chunks mentioning 'castle' and 'rook' are preferred.


,qid,base_rank,rerank_rank,shift,chunk_id,page,cosine,ce_score,gold,text
0,L2,1,3,-2,43,13,0.695,6.46,,"4.6.3 If an opponent's piece stands on the square of promotion, it must be captured. In this case the player m…"
1,L2,2,1,1,41,13,0.652,6.99,,4.4 If a player having the move: 4.4.1 touches his/her king and a rook he/she must castle on that side if it i…
2,L2,3,5,-2,32,11,0.608,6.04,★,"3.8.2.1 The right to castle has been lost: 3.8.2.1.1 If the king has already moved, or 3.8.2.1.2 With a rook t…"
3,L2,4,2,2,44,13,0.575,6.84,,rook (but see 4.4.2). It may happen that a player moves his/her king to the final castle square but doesn't mo…
4,L2,5,4,1,203,46,0.575,6.30,,"Chess960 castling rules II.3.1 Chess960 allows each player to castle once per game, a move by potentially both…"
5,L2,6,7,-1,207,47,0.571,2.79,,"to state ""I am about to castle"" before castling. 2. In some starting positions, the king or rook (but not both…"
6,L2,7,8,-1,204,46,0.528,1.71,,"castling, because the regular rules presume initial locations of the rook and king that are often not applicab…"
7,L2,8,6,2,122,28,0.525,5.57,,"rights with a rook that has not been moved, but forfeited these after moving. The castling rights are lost onl…"
8,L2,9,10,-1,33,11,0.517,-1.07,,"by one or more of the opponent's pieces, or 3.8.2.2.2 if there is any piece between the king and the rook with…"
9,L2,10,9,1,45,13,0.507,1.18,,"as a rook, even if he/she names it as a ""queen"" or any other piece. If he/she moves the upside-down rook diago…"


## 8. Answer 1 vs. Answer 2
Both answers use the identical prompt template; only the three context chunks differ.

In [12]:
# Run both full pipelines (retrieval -> [re-rank] -> Gemini) for every test query and print the answers side by side.
answers = []
for fq in FAILURE_QUERIES:
    p1 = pipeline_1(fq["query"])
    p2 = pipeline_2(fq["query"])
    answers.append({"qid": fq["qid"], "query": fq["query"],
                    "context_1": p1["context_ids"], "answer_1": p1["answer"],
                    "context_2": p2["context_ids"], "answer_2": p2["answer"],
                    # timings kept for the end-to-end latency table in section 9
                    "P1 retrieve ms": 1000 * p1["t_retrieve"], "P1 LLM ms": 1000 * p1["t_llm"],
                    "P2 retrieve+rerank ms": 1000 * (p2["t_retrieve"] + p2["t_rerank"]),
                    "P2 LLM ms": 1000 * p2["t_llm"]})
    print("=" * 110)
    print(f"{fq['qid']} [{fq['category']}] {fq['query']}")
    print(f"\n  Answer 1 (baseline, chunks {p1['context_ids']}):\n  {p1['answer']}")
    print(f"\n  Answer 2 (re-ranked, chunks {p2['context_ids']}):\n  {p2['answer']}\n")

with open(RESULTS / "answers.json", "w", encoding="utf-8") as f:
    json.dump(answers, f, indent=2, ensure_ascii=False)

Q1 [Keyword trap / distractor] How much extra time does the opponent get when a player makes an illegal move?

  Answer 1 (baseline, chunks [157, 164, 100]):
  According to Article A.6, the penalty for an illegal move "is the addition of one minute to the opponent, instead of two minutes" as applied in standard chess. Additionally, Article A.3 specifies that the penalties mentioned in Articles 7 and 9 of the Competitive Rules of Play "shall be one minute instead of two minutes." Therefore, the opponent receives one minute of extra time under these rules, rather than the standard two minutes.

  Answer 2 (re-ranked, chunks [97, 129, 61]):
  According to Article 7.5.5, the arbiter shall give two minutes of extra time to the opponent for the first completed illegal move made by a player. For a second completed illegal move by the same player, no additional time is granted; instead, the arbiter shall declare the game lost by that player, provided the opponent can checkmate by a possible se

Q2 [Exception (rule stated elsewhere)] What is the penalty for an illegal move in rapid chess?

  Answer 1 (baseline, chunks [100, 164, 34]):
  According to Article A.6, the penalty for an illegal move is the addition of one minute to the opponent's clock, rather than two minutes as in standard chess. Under this rule, a player does not lose the game upon their first illegal move, but only with the second illegal move.

  Answer 2 (re-ranked, chunks [100, 164, 156]):
  In Rapid chess, "the player does not lose the game with the first illegal move, but only with the second, as it is in standard chess" (Article A.6). The penalty awarded is "the addition of one minute to the opponent, instead of two minutes" (Article A.6). Additionally, Article A.3 states that the penalties mentioned in Articles 7 and 9 of the Competitive Rules of Play shall be one minute instead of two minutes.



Q3 [Negation / role limit] Which things is the arbiter not allowed to tell a player during the game?

  Answer 1 (baseline, chunks [134, 138, 153]):
  The provided context does not answer this.

  Answer 2 (re-ranked, chunks [153, 152, 135]):
  According to the provided context, the arbiter "shall not indicate the number of moves completed, except in applying Article 8.5 when at least one flag has fallen." Additionally, the arbiter must "refrain from informing a player that his/her opponent has completed a move or that the player has not pressed his/her clock."



Q4 [Exception / corner case] Is a player with less than five minutes required to keep recording the moves?

  Answer 1 (baseline, chunks [157, 156, 67]):
  The provided context does not answer this. While the passages discuss scorekeeping rules for Rapid chess under Appendix A (specifically Article A.2, which notes players do not need to record moves), they contain no rules regarding scorekeeping requirements when a player has less than five minutes remaining.

  Answer 2 (re-ranked, chunks [105, 159, 156]):
  According to Article 8.4, if a player has less than five minutes left on their clock during an allotted period of time and does not have an additional time of 30 seconds or more added with each move, they are not obliged to meet the requirements of Article 8.1.1. Therefore, for the remainder of that period, the player is not required to keep recording the moves unless they receive an increment of 30 seconds or more per move.



Q5 [Corner case (buried at rank 5)] If I use two hands to castle illegally, is that counted as two illegal moves?

  Answer 1 (baseline, chunks [41, 34, 47]):
  The provided context does not answer this. While Articles 3.10, 4.4, and 4.7–4.8 discuss legal and illegal moves, illegal positions, and castling procedures, they do not contain any rules regarding using two hands or whether doing so constitutes two illegal moves.

  Answer 2 (re-ranked, chunks [99, 92, 100]):
  No, it is not counted as two illegal moves. According to the provided context, "when there are two (2) illegal moves in one move (for example illegal castling made by two hands...), they count as one (1) illegal move and the player shall not be forfeited, unless it is the second such transgression." Furthermore, Article 7.5.4 specifies that using two hands to make a single move and pressing the clock is considered and penalised as if an illegal move.



Q6 [Negation / constraint] Can an illegal move in rapid chess be corrected after the opponent has replied?

  Answer 1 (baseline, chunks [91, 35, 34]):
  The provided context does not answer this. While the passages mention when an arbiter intervenes in Rapid chess (Passage 2) and how illegal moves are handled generally under Article 7.5.1, they do not state whether an illegal move in Rapid chess can be corrected after an opponent has already replied.

  Answer 2 (re-ranked, chunks [162, 35, 97]):
  According to the provided context, once the opponent has made his/her next move, an illegal move cannot be corrected unless this is agreed by the players without the intervention of the arbiter (Passage 1, preceding Article A.5.3). If the opponent does not claim and the arbiter does not intervene before the reply is made, the illegal move stands and the game continues. Therefore, it can only be corrected after the opponent's reply if both players mutually agree to do so without arbiter invo

Q7 [Negation / prohibition] Can I keep my phone in my bag during the game?

  Answer 1 (baseline, chunks [143, 142, 139]):
  Generally, no mobile phone is allowed in the playing venue, whether switched on or off, and having one found with a player results in an immediate loss of the game (Passage 3, following Article 11.5). However, organizers may specify in the regulations of the event that a player is permitted to bring such a device if it is completely switched off before entering the playing area and stored in a separate bag (Passage 1 and Passage 2). Under these allowed conditions, the bag must not be in contact with the player, and the player must not have access to it during the game without the arbiter's permission (Passage 2).

  Answer 2 (re-ranked, chunks [143, 142, 135]):
  According to Article 11.3.2, a player is generally forbidden from having any electronic device in the playing venue unless specifically approved by the arbiter. However, under Article 11.3.2.1, the regul

L1 [Limitation: article text demoted] Can a player write down his move on the scoresheet before playing it?

  Answer 1 (baseline, chunks [103, 107, 105]):
  No, a player is generally not allowed to record a move before playing it. According to Article 8.1.2, "It is forbidden to record the moves in advance, unless the player is claiming a draw according to Article 9.2, or 9.3 or adjourning a game according to Guidelines I.1.1". Therefore, writing down a move prior to playing it is only permitted in the specific cases of making a draw claim or adjourning the game.

  Answer 2 (re-ranked, chunks [106, 112, 109]):
  No, it is generally forbidden for a player to record a move before playing it, even if the opponent has only one legal move. The only exceptions are in the case of a draw claim (Article 9.2. and 9.3) and adjourning, where doing so is allowed.



L2 [Limitation: re-ranker demotes gold] Can a player castle with a rook that has already moved?

  Answer 1 (baseline, chunks [43, 41, 32]):
  No, a player cannot castle with a rook that has already moved. According to Article 3.8.2.1 and Article 3.8.2.1.2, the right to castle has been lost "[w]ith a rook that has already moved." Therefore, castling with that rook is no longer legally permitted.

  Answer 2 (re-ranked, chunks [41, 44, 43]):
  The provided context does not answer this. While the excerpts discuss touched pieces and procedures for when castling is illegal under Article 4.4 (such as Articles 4.4.1, 4.4.2, and 4.4.3), they do not provide the specific rules regarding whether a rook that has already moved can be used for castling.



## 9. Latency profiling
Retrieval-only latencies are averaged over many repetitions after a warm-up call (CPU, batch size 1 query).
LLM latency is measured separately on the failure queries because it is network-bound and dominates end-to-end time.

In [13]:
import platform, torch
print(platform.processor(), "| torch threads:", torch.get_num_threads())

# Time only the retrieval stages (no LLM) many times, so the means/percentiles are stable.
N_REPEATS = 20
queries = [fq["query"] for fq in FAILURE_QUERIES]
pipeline_1(queries[0], call_llm=False); pipeline_2(queries[0], call_llm=False)   # warm-up

lat = {"p1_retrieve": [], "p2_retrieve": [], "p2_rerank": []}
for _ in range(N_REPEATS):
    for q in queries:
        r1 = pipeline_1(q, call_llm=False)
        r2 = pipeline_2(q, call_llm=False)
        lat["p1_retrieve"].append(r1["t_retrieve"])
        lat["p2_retrieve"].append(r2["t_retrieve"])
        lat["p2_rerank"].append(r2["t_rerank"])

ms = lambda xs: (1000 * statistics.mean(xs), 1000 * statistics.median(xs), 1000 * np.percentile(xs, 95))
p1 = np.array(lat["p1_retrieve"]); p2 = np.array(lat["p2_retrieve"]) + np.array(lat["p2_rerank"])
latency = pd.DataFrame([
    ["Pipeline 1: bi-encoder query encode + FAISS top-10", *ms(lat["p1_retrieve"])],
    ["Pipeline 2: bi-encoder + FAISS top-10",               *ms(lat["p2_retrieve"])],
    ["Pipeline 2: cross-encoder re-rank of 10 pairs",       *ms(lat["p2_rerank"])],
    ["Pipeline 1 total retrieval",                          *ms(p1)],
    ["Pipeline 2 total retrieval (+re-rank)",               *ms(p2)],
], columns=["stage", "mean ms", "median ms", "p95 ms"]).round(2)
print(f"{len(lat['p1_retrieve'])} timed runs per stage; re-ranking overhead = "
      f"{statistics.mean(p2) / statistics.mean(p1):.1f}x the baseline retrieval time")
latency

Intel64 Family 6 Model 186 Stepping 2, GenuineIntel | torch threads: 14


180 timed runs per stage; re-ranking overhead = 17.0x the baseline retrieval time


,stage,mean ms,median ms,p95 ms
0,Pipeline 1: bi-encoder query encode + FAISS top-10,38.27,36.14,47.64
1,Pipeline 2: bi-encoder + FAISS top-10,37.07,35.37,45.67
2,Pipeline 2: cross-encoder re-rank of 10 pairs,615.11,608.51,732.07
3,Pipeline 1 total retrieval,38.27,36.14,47.64
4,Pipeline 2 total retrieval (+re-rank),652.18,646.15,772.78


In [14]:
# End-to-end timing including the LLM call (timings recorded while generating the answers in section 8).
if llm_client is not None:
    e2e = pd.DataFrame(answers)[["qid", "P1 retrieve ms", "P1 LLM ms", "P2 retrieve+rerank ms", "P2 LLM ms"]]
    e2e.insert(3, "P1 total ms", e2e["P1 retrieve ms"] + e2e["P1 LLM ms"])
    e2e["P2 total ms"] = e2e["P2 retrieve+rerank ms"] + e2e["P2 LLM ms"]
    e2e = e2e.set_index("qid").round(1)
    e2e.loc["mean"] = e2e.mean(numeric_only=True).round(1)
    e2e.to_csv(RESULTS / "latency_end_to_end.csv")
    display(e2e)
else:
    print("Skipped: no GEMINI_API_KEY.")
latency.to_csv(RESULTS / "latency_retrieval.csv", index=False)

,P1 retrieve ms,P1 LLM ms,P1 total ms,P2 retrieve+rerank ms,P2 LLM ms,P2 total ms
qid,,,,,,
Q1,37.4,25040.5,25078.0,692.7,31223.3,31916.0
Q2,42.6,72358.6,72401.2,541.4,8001.2,8542.7
Q3,49.2,28737.1,28786.4,671.9,3505.8,4177.7
Q4,40.9,4539.4,4580.3,796.6,4087.4,4884.0
Q5,174.9,4918.4,5093.3,648.2,5472.1,6120.3
Q6,38.2,6534.5,6572.7,617.1,8302.6,8919.7
Q7,43.2,12266.2,12309.4,540.6,18249.7,18790.3
L1,32.7,23493.2,23525.9,581.8,30119.3,30701.1
L2,34.5,83473.0,83507.5,729.0,59286.0,60015.0


## 10. Scaling: why not cross-encode the whole collection?
To give the scaling question numbers, the cross-encoder is run against **every** chunk for one query and compared with
the bi-encoder path. The bi-encoder's document vectors are computed once offline; at query time only the query is
encoded. The cross-encoder has no reusable document representation, so each query needs one full transformer pass
per chunk.

In [15]:
# Time the bi-encoder search vs. cross-encoding EVERY chunk for one query, then extrapolate to larger corpora.
q = FAILURE_QUERIES[0]["query"]
t0 = time.perf_counter(); baseline_retrieve(q, k=10); t_bi = time.perf_counter() - t0

t0 = time.perf_counter()
full_scores = cross_encoder.predict([(q, t) for t in chunk_texts], batch_size=64, show_progress_bar=False)
t_ce_all = time.perf_counter() - t0
per_pair = t_ce_all / len(chunk_texts)

full_rank = np.argsort(-full_scores)[:10].tolist()
print(f"Bi-encoder search over {len(chunks)} chunks : {1000 * t_bi:8.1f} ms")
print(f"Cross-encoder over all {len(chunks)} chunks  : {1000 * t_ce_all:8.1f} ms  ({1000 * per_pair:.2f} ms per pair)")
print(f"Top-10 by full cross-encoding: {full_rank}")
print(f"Top-10 by two-stage pipeline : {[c['chunk_id'] for c in detail['Q1']]}")

# Cost of full cross-encoding grows linearly with corpus size; two-stage cost stays at 10 pairs per query.
scale = pd.DataFrame({"collection size (chunks)": [238, 10_000, 1_000_000, 100_000_000]})
scale["cross-encode everything (s/query)"] = (scale.iloc[:, 0] * per_pair).round(1)
scale["two-stage: re-rank 10 (s/query)"] = round(10 * per_pair, 3)
scale

Bi-encoder search over 238 chunks :     43.7 ms
Cross-encoder over all 238 chunks  :  17284.1 ms  (72.62 ms per pair)
Top-10 by full cross-encoding: [97, 221, 211, 129, 61, 101, 164, 226, 210, 100]
Top-10 by two-stage pipeline : [97, 129, 61, 164, 100, 91, 99, 92, 157, 34]


,collection size (chunks),cross-encode everything (s/query),two-stage: re-rank 10 (s/query)
0,238,17.3,0.726
1,10000,726.2,0.726
2,1000000,72622.4,0.726
3,100000000,7262244.5,0.726
